### **Purpose**

This Python script is designed to:

- Remove **encryption** from PDFs
- Flatten **layers**
- Remove interactive **digital signature fields**
- Strip **security headers**

### **Result**

Generates a **clean, upload-ready PDF** that works on platforms rejecting files with security restrictions or multiple layers.

- **Secure & Offline**: No data is uploaded anywhere—ideal for privacy-conscious users.
- Removing a signature field also removes the original digital signature's verification; the output is no longer digitally signed.
- Perfect for documents like **bank statements**, **payslips**, **agreements**, and more.

In [5]:
# Run this command to install PyPDF2 if you haven't already:
%pip install PyPDF2

Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [6]:
from PyPDF2 import PdfReader, PdfWriter


def is_signature_widget(annotation):
    if annotation.get("/Subtype") != "/Widget":
        return False

    current = annotation
    while current is not None:
        current = current.get_object()
        if current.get("/FT") == "/Sig":
            return True

        signature_value = current.get("/V")
        if signature_value is not None:
            signature_value = signature_value.get_object()
            if (
                hasattr(signature_value, "get")
                and signature_value.get("/ByteRange") is not None
            ):
                return True

        current = current.get("/Parent")

    return False


def remove_signature_widgets(page):
    annotations = page.get("/Annots")
    if annotations is None:
        return

    annotations = annotations.get_object()
    for index in reversed(range(len(annotations))):
        annotation = annotations[index].get_object()
        if is_signature_widget(annotation):
            del annotations[index]

In [7]:
input_path = "/Users/rismalik/Downloads/EAadhaar_2728047901628520210118143503_2511202619331.pdf"
output_path = "/Users/rismalik/Downloads/Document_Cleaned.pdf"

In [8]:
# Open the original PDF in read-binary mode
with open(input_path, "rb") as infile:
    reader = PdfReader(infile)

    # If the PDF is actually password-protected, you'd need:
    reader.decrypt("RAJE1961")

    writer = PdfWriter()

    # Copy each page into a new writer
    for page in reader.pages:
        writer.add_page(page)
        remove_signature_widgets(writer.pages[-1])

    # Write everything out to a brand new PDF
    with open(output_path, "wb") as outfile:
        writer.write(outfile)
    print("Done! Clean file saved as:", output_path)

Done! Clean file saved as: /Users/rismalik/Downloads/Document_Cleaned.pdf
